# Week 2 – Logistics Data Collection, Cleaning and Preprocessing
This notebook demonstrates the complete preprocessing pipeline using the Week 2 logistics dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv('../data/logistics_raw_data.csv')
df.head()

## Data Quality Checks

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
df.describe(include='all')

## Cleaning

In [ ]:
for col in ['Product_Category','Origin','Destination','Transport_Mode','Delivery_Status']:
    df[col] = df[col].astype('string').str.strip()
df['Product_Category'] = df['Product_Category'].str.title()
df['Transport_Mode'] = df['Transport_Mode'].str.title()
df = df.drop_duplicates()
numeric_cols = ['Quantity','Shipping_Cost_INR','Shipping_Time_Days','Inventory_Level']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[df['Shipping_Cost_INR'] <= 0, 'Shipping_Cost_INR'] = np.nan
df.loc[df['Shipping_Time_Days'] <= 0, 'Shipping_Time_Days'] = np.nan
df.loc[df['Inventory_Level'] < 0, 'Inventory_Level'] = np.nan

## Outlier Treatment and Missing-Value Imputation

In [ ]:
def iqr_cap(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return series.clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
for col in numeric_cols:
    df[col] = iqr_cap(df[col])
    df[col] = df[col].fillna(df[col].median())
df.to_csv('../data/logistics_cleaned_data.csv', index=False)
df.head()

## Normalization and Encoding

In [ ]:
scaler = MinMaxScaler()
normalized = pd.DataFrame(scaler.fit_transform(df[numeric_cols]),
                          columns=[c + '_Normalized' for c in numeric_cols])
encoded = pd.get_dummies(df[['Product_Category','Transport_Mode','Delivery_Status']],
                         prefix=['Category','Mode','Status'], dtype=int)
preprocessed = pd.concat([df, normalized, encoded], axis=1)
preprocessed.to_csv('../data/logistics_preprocessed_data.csv', index=False)
preprocessed.head()